In [ ]:
import os
import io
import time
import subprocess
import logging
from datetime import datetime
import pandas as pd
import requests

# Ensure essential structures are updated and patched silently
subprocess.run(["pip", "install", "pandas", "openpyxl", "requests", "-q"], check=False)

# ═══════════════════════════════════════════════════════════
#  SETTINGS & PERFORMANCE MATRIX AUDIT
# ═══════════════════════════════════════════════════════════
PARAMS = {
    "VIRTUAL_START_CAPITAL": 20000,   # Local mock USDT bankroll pool
    "ALLOCATION_PER_TRADE": 5000,     # Notional size split per arb cycle ($2500 Spot / $2500 Perp)
    "MIN_ANNUAL_YIELD_TRIGGER": 0.12, # 12% annualized minimal target rate to trigger entry
    "EXECUTION_FEE_ROUNDTRIP": 0.0016,# Combined spot/perp slippage + transaction friction (0.16%)
    "SCAN_INTERVAL_SEC": 3,           # Time elapsed between real-time data frame requests
    "MAX_RUN_CYCLES": 15,             # Total polling cycles to loop before generating excel logs
    "EXCEL_FILE": "funding_arb_results.xlsx"
}

# Targeted liquid crypto assets to track
MONITORED_ASSETS = ["BTC", "ETH", "SOL", "BNB"]

log_capture_string = io.StringIO()
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(), logging.StreamHandler(log_capture_string)]
)
log = logging.getLogger("FundingArb")

# ═══════════════════════════════════════════════════════════
#  NETWORK DATA SCRAPER LAYER (ANTI-BLOCK BYPASS)
# ═══════════════════════════════════════════════════════════
def get_live_market_data():
    """Pulls public spot prices and live futures funding metrics through API mirrors."""
    spot_url = "https://api.binance.com/api/v3/ticker/price"
    perp_url = "https://fapi.binance.com/fapi/v1/premiumIndex"
    
    # Regional block fallback gateway alternatives
    fallbacks = [
        ("https://api1.binance.com/api/v3/ticker/price", "https://fapi1.binance.com/fapi/v1/premiumIndex"),
        ("https://api3.binance.com/api/v3/ticker/price", "https://fapi3.binance.com/fapi/v1/premiumIndex")
    ]
    
    for s_url, p_url in [(spot_url, perp_url)] + fallbacks:
        try:
            s_res = requests.get(s_url, timeout=4).json()
            p_res = requests.get(p_url, timeout=4).json()
            
            spot_map = {item['symbol'].replace("USDT", ""): float(item['price']) for item in s_res if "USDT" in item['symbol']}
            perp_map = {item['symbol'].replace("USDT", ""): {
                "funding_rate": float(item['lastFundingRate']),
                "mark_price": float(item['markPrice'])
            } for item in p_res if "USDT" in item['symbol'] and 'lastFundingRate' in item}
            
            return spot_map, perp_map
        except Exception:
            continue
    raise ConnectionError("Unable to cross past regional DNS blocks to resolve endpoints.")

# ═══════════════════════════════════════════════════════════
#  SYSTEM DASHBOARD RENDER PIPELINE
# ═══════════════════════════════════════════════════════════
def print_dashboard(wallet, active_positions, asset_metrics, cycle):
    os.system("cls" if os.name == "nt" else "clear")
    print("╔══════════════════════════════════════════════════════════╗")
    print("║ ♟ KEYLESS FUNDING RATE DELTA-NEUTRAL ARBITRAGE SYSTEM     ║")
    print(f"║ Cycle Trace: {cycle:>3}/{PARAMS['MAX_RUN_CYCLES']}  |  Engine Update Time: {datetime.now().strftime('%H:%M:%S')}         ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print(f"║ Total Capital Account Balance: ${wallet['total_equity']:<12,.2f} USDT              ║")
    print(f"║ In-Memory Free Cash Reserves:  ${wallet['cash']:<12,.2f} USDT              ║")
    print(f"║ Currently Locked Open Hedges:  {len(active_positions):<2}                          ║")
    print("╠══════════════════════════════════════════════════════════╣")
    print("║ LIVE YIELD MARKET SCANNER                                ║")
    print("║ Token     Spot Price    Funding Rate    Est. Annualized  ║")
    print("╠══════════════════════════════════════════════════════════╣")
    
    for asset in MONITORED_ASSETS:
        if asset not in asset_metrics: continue
        m = asset_metrics[asset]
        ann_yield = m['funding_rate'] * 3 * 365 * 100  # Convert 8h payment rate to APR %
        
        status_flag = " [ACTIVE]" if asset in active_positions else ""
        print(f"║ {asset:<5}    ${m['spot']:<10,.2f}   {m['funding_rate']:>+.5f}         {ann_yield:>6.2f}%{status_flag:<9} ║")
    print("╚══════════════════════════════════════════════════════════╝")

# ═══════════════════════════════════════════════════════════
#  CORE ARBITRAGE RUNNER ENGINE
# ═══════════════════════════════════════════════════════════
def run_funding_bot():
    log.info("Booting keyless live funding engine manager...")
    
    wallet = {
        "cash": PARAMS["VIRTUAL_START_CAPITAL"],
        "total_equity": PARAMS["VIRTUAL_START_CAPITAL"]
    }
    
    active_positions = {}
    arbitrage_ledger = []
    
    for cycle in range(1, PARAMS["MAX_RUN_CYCLES"] + 1):
        try:
            spot_data, perp_data = get_live_market_data()
            
            asset_metrics = {}
            for asset in MONITORED_ASSETS:
                if asset in spot_data and asset in perp_data:
                    asset_metrics[asset] = {
                        "spot": spot_data[asset],
                        "funding_rate": perp_data[asset]["funding_rate"],
                        "mark_price": perp_data[asset]["mark_price"]
                    }
                    
            # Evaluate exits on currently held positions first
            for asset in list(active_positions.keys()):
                pos = active_positions[asset]
                current_rate = asset_metrics[asset]["funding_rate"]
                annual_apr = current_rate * 3 * 365
                
                # Exit when the premium falls below minimal target bounds
                if annual_apr < (PARAMS["MIN_ANNUAL_YIELD_TRIGGER"] * 0.5):
                    log.info(f"[{asset}] Yield evaporated to {annual_apr*100:.2f}%. Closing neutral position matrix...")
                    
                    # Compute settlement values relative to entry benchmarks
                    spot_unwind = asset_metrics[asset]["spot"] * pos["units"]
                    perp_unwind = (pos["entry_perp"] - asset_metrics[asset]["mark_price"]) * pos["units"] + (PARAMS["ALLOCATION_PER_TRADE"] * 0.5)
                    
                    gross_return = spot_unwind + perp_unwind
                    friction = PARAMS["ALLOCATION_PER_TRADE"] * PARAMS["EXECUTION_FEE_ROUNDTRIP"]
                    net_settled_cash = gross_return - friction
                    
                    wallet["cash"] += net_settled_cash
                    trade_pnl = net_settled_cash - PARAMS["ALLOCATION_PER_TRADE"]
                    
                    arbitrage_ledger.append({
                        "Timestamp": datetime.now().strftime("%H:%M:%S"),
                        "Asset": asset,
                        "Type": "HARVEST_EXIT",
                        "Captured Yield ($)": round(trade_pnl, 4),
                        "Rate At Exit": f"{current_rate:+.5f}",
                        "Account Standing ($)": round(wallet["cash"], 2)
                    })
                    del active_positions[asset]

            # Evaluate strategy entries
            for asset, metrics in asset_metrics.items():
                if asset in active_positions: continue
                
                annual_apr = metrics["funding_rate"] * 3 * 365
                
                # High funding rate detected -> Enter Short Perp / Long Spot
                if annual_apr >= PARAMS["MIN_ANNUAL_YIELD_TRIGGER"]:
                    if wallet["cash"] < PARAMS["ALLOCATION_PER_TRADE"]: continue
                    
                    log.info(f"🔥 OPPORTUNITY FOUND! {asset} Annualized Funding Yield: {annual_apr*100:.2f}%")
                    
                    wallet["cash"] -= PARAMS["ALLOCATION_PER_TRADE"]
                    units_purchased = (PARAMS["ALLOCATION_PER_TRADE"] * 0.5) / metrics["spot"]
                    
                    active_positions[asset] = {
                        "entry_spot": metrics["spot"],
                        "entry_perp": metrics["mark_price"],
                        "units": units_purchased,
                        "allocated_capital": PARAMS["ALLOCATION_PER_TRADE"]
                    }
                    
                    arbitrage_ledger.append({
                        "Timestamp": datetime.now().strftime("%H:%M:%S"),
                        "Asset": asset,
                        "Type": "HEDGE_ENTRY",
                        "Captured Yield ($)": 0.0,
                        "Rate At Exit": f"{metrics['funding_rate']:+.5f}",
                        "Account Standing ($)": round(wallet["cash"], 2)
                    })

            # Simulate interval check: Credit funding rate fees into your wallet balance rows
            for asset, pos in active_positions.items():
                live_rate = asset_metrics[asset]["funding_rate"]
                # In long funding intervals, Shorts get paid by Longs
                accrued_funding_payment = (pos["allocated_capital"] * 0.5) * live_rate
                
                wallet["cash"] += accrued_funding_payment
                if accrued_funding_payment != 0:
                    log.info(f"💰 [FEE CASH-IN] Captured ${accrued_funding_payment:+.4f} from {asset} Perp funding interval tick.")

            # Update portfolio mark-to-market valuations
            running_positions_valuation = 0
            for asset, pos in active_positions.items():
                s_val = asset_metrics[asset]["spot"] * pos["units"]
                p_val = (pos["entry_perp"] - asset_metrics[asset]["mark_price"]) * pos["units"] + (pos["allocated_capital"] * 0.5)
                running_positions_valuation += (s_val + p_val)
                
            wallet["total_equity"] = wallet["cash"] + running_positions_valuation
            
            print_dashboard(wallet, active_positions, asset_metrics, cycle)
            time.sleep(PARAMS["SCAN_INTERVAL_SEC"])

        except Exception as e:
            log.error(f"Cycle iteration processing exception: {e}")
            time.sleep(3)

    # ═══════════════════════════════════════════════════════════
    #  EXCEL PIPELINE EXPORT AUDITING LAYER
    # ═══════════════════════════════════════════════════════════
    log.info("Scan cycles executed successfully. Saving session logs and configurations directly to Excel sheets...")
    
    summary_df = pd.DataFrame(arbitrage_ledger) if arbitrage_ledger else pd.DataFrame(columns=["Timestamp", "Asset", "Type", "Captured Yield ($)"])
    param_df = pd.DataFrame(list(PARAMS.items()), columns=["Applied Strategy Setting", "Value Metric"])
    log_df = pd.DataFrame(log_capture_string.getvalue().split('\n'), columns=["Live Execution Running Logs"])
    
    with pd.ExcelWriter(PARAMS["EXCEL_FILE"], engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Arbitrage_Trade_Ledger", index=False)
        param_df.to_excel(writer, sheet_name="Strategy_Parameters", index=False)
        log_df.to_excel(writer, sheet_name="System_Run_Logs", index=False)
        
    print("\n" + "═"*65)
    print(" LIVE FUNDING ARBITRAGE TESTING COMPLETE")
    print("═"*65)
    print(f" Workbook ledger exported to target local root path -> {PARAMS['EXCEL_FILE']}")
    print(" Tabs updated: 'Arbitrage_Trade_Ledger', 'Strategy_Parameters', 'System_Run_Logs'")
    print("═"*65)

if __name__ == "__main__":
    run_funding_bot()